In [21]:
import os
import subprocess
from pathlib import Path

from anthropic import Anthropic
from anthropic.types import Message, MessageParam, TextBlock
from dotenv import dotenv_values

env_path = Path.cwd().parent / ".env.template"
for key, ref in dotenv_values(env_path).items():
    if ref is None:
        continue
    os.environ[key] = subprocess.run(
        ["op", "read", ref], capture_output=True, text=True, check=True
    ).stdout.strip()

client = Anthropic(
    base_url="https://openrouter.ai/api",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
model = "deepseek/deepseek-v4.1-flash"

In [22]:
def get_reply(message: Message) -> str:
    return next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "",
    )


def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages: list[MessageParam]) -> str:
    message: Message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return get_reply(message)


# Start with an empty message list
messages: list[MessageParam] = []